# Dreamer checkpoint inspector

Load a model snapshot and a recorded rollout, then inspect the exact states and action distributions used by the policy. Playback is a self-contained HTML viewer; it needs no widget extension or internet connection.

The default paths use a small **untrained demonstration** prepared locally to try the controls. To inspect training, create a recording from one of your saved checkpoints and change `recording_path` below.

From the repository root:

```bash
uv run python scripts/inspect_dreamer.py \
  --checkpoint experiments/dreamer/checkpoints/RUN_ID/00000010 \
  --output experiments/dreamer/inspections/RUN_ID_00000010 \
  --num-episodes 2 --max-steps 1000
```

Training saves the initial model, snapshots before evaluations, and the final model under `--checkpoint-dir`. These snapshots contain models and configuration for inspection, without optimizer or replay state.


In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import display

from dreamer.checkpoints import load_checkpoint
from dreamer.inspection import (
    imagine_from_frame,
    load_recording,
    predict_recorded_actions,
)
from dreamer.viewer import view_imagination, view_prediction, view_recording

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

run_id = "lv7symv9"
recording_path = project_root / "experiments/dreamer/inspections" / f"{run_id}_00016000"

In [ ]:
episodes, recording_metadata = load_recording(recording_path)
checkpoint_path = Path(recording_metadata["checkpoint"])
# If you copied a recording from another machine, set its local checkpoint path here.
models, checkpoint_metadata = load_checkpoint(checkpoint_path)

print("Checkpoint:", checkpoint_path)
print("Training step:", checkpoint_metadata["train_step"])
print(
    "Training wall time:",
    round(checkpoint_metadata.get("elapsed_seconds", 0) / 60, 1),
    "minutes",
)
print("Recorded episodes:", len(episodes))
print("Recording took:", round(recording_metadata["elapsed_seconds"], 1), "seconds")

## Recorded behaviour

Select an episode, play it, or move to a particular frame. Each row is a decision state: observation `o_t`, the posterior state used by the actor, action `a_t`, and the following observation/reward.

- The observation reward plot compares incoming `r_t` with its prediction from the posterior. The table also shows `r_(t+1)`, produced by the selected action.
- Action curves are the actual **truncated-normal densities**. The distribution mean can differ from its underlying normal's location parameter.
- World-model losses are computed from the recorded latent samples using the existing class loss function. No new posterior is sampled to make the plots.
- The recorded discounted return uses a zero tail at the end of the recording. It is a reference, not an unbiased critic target at a time limit.
- Orange outlines in the latent heatmaps mark the categorical classes that were sampled.


In [ ]:
display(view_recording(episodes, recording_metadata))


## Branch from a selected posterior state

The playback viewer displays the `episode_index` and `frame` to enter below. Change those values and run the branch cells. The recording remains fixed; a branch samples a new imagined future without changing the loaded checkpoint.

The seed controls both action sampling and the world model's categorical sampling, so rerunning the same branch is reproducible.


In [ ]:
episode_index = 0
frame = 50
horizon = 15
seed = 20

episode = episodes[episode_index]


### Policy imagination

The actor chooses new actions from the recorded posterior start. The real future is shown only as a reference because its sampled actions can differ. The loss curves show per-step contributions to the actor and critic objectives, including lambda returns and discount weights.

The return-contribution plot splits the lambda return into reward and critic-value terms using the current lambda. The reward contribution therefore retains lambda weighting; it is not a Monte Carlo return.


In [ ]:
branch = imagine_from_frame(models, episode, frame, horizon=horizon, seed=seed)
display(view_imagination(branch))

print("Mean actor loss:", float(branch["actor_loss"].mean()))
print("Mean critic loss:", float(branch["critic_loss"].mean()))
print("Imagined rewards:", np.round(branch["predicted_reward"], 3))


### Dynamics under recorded actions

This branch uses the exact actions from the recording. Predicted images and rewards can therefore be compared with the real trajectory. It uses the posterior only for the starting state; subsequent states come from the transition prior.

The viewer's frame 0 is the **first predicted next state**, unlike policy imagination, whose frame 0 is the starting state. This makes each prediction line up with its actual next observation and reward.


In [ ]:
prediction = predict_recorded_actions(
    models, episode, frame, horizon=horizon, seed=seed
)
display(view_prediction(prediction))

print("Image prediction losses:", np.round(prediction["image_loss"], 3))
print("Predicted rewards:", np.round(prediction["predicted_reward"], 3))
print("Actual rewards:", np.round(prediction["rewards"], 3))


## What to check first

1. At positive reward events, does the reward model predict their magnitude and timing?
2. When the car stalls or brakes, does the policy distribution explain the chosen action?
3. Under recorded actions, how quickly do the prior's images and rewards diverge from reality?
4. In policy imagination, do the predicted return and its losses mostly reflect immediate rewards or critic values?
5. Do posterior entropy, prior entropy, or KL change around the point where real behaviour deteriorates?

Keep a recording paired with the checkpoint that generated its states. To compare another checkpoint, collect its own recording rather than feeding it the first model's hidden states.
